In [1]:
df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("Files/healthcare_dataset.csv")

display(df_raw)

StatementMeta(, b4b70097-a892-4895-9852-0e8a23690dbb, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8b53f8d9-d2e0-4b90-b0eb-e8906e9615c3)

In [2]:
print("Row Count:", df_raw.count())
print("Column Count:", len(df_raw.columns))

StatementMeta(, b4b70097-a892-4895-9852-0e8a23690dbb, 4, Finished, Available, Finished, False)

Row Count: 55500
Column Count: 15


In [4]:
from pyspark.sql.functions import col

df_bronze = df_raw.select(
    col("Name").alias("Name"),
    col("Age").alias("Age"),
    col("Gender").alias("Gender"),
    col("Blood Type").alias("Blood_Type"),
    col("Medical Condition").alias("Medical_Condition"),
    col("Date of Admission").alias("Date_of_Admission"),
    col("Doctor").alias("Doctor"),
    col("Hospital").alias("Hospital"),
    col("Insurance Provider").alias("Insurance_Provider"),
    col("Billing Amount").alias("Billing_Amount"),
    col("Room Number").alias("Room_Number"),
    col("Admission Type").alias("Admission_Type"),
    col("Discharge Date").alias("Discharge_Date"),
    col("Medication").alias("Medication"),
    col("Test Results").alias("Test_Results")
)

StatementMeta(, b4b70097-a892-4895-9852-0e8a23690dbb, 6, Finished, Available, Finished, False)

In [6]:
display(df_bronze)

StatementMeta(, b4b70097-a892-4895-9852-0e8a23690dbb, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c44aa1fc-97e6-408b-a796-af64854d103d)

In [8]:
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_healthcare_raw")

StatementMeta(, b4b70097-a892-4895-9852-0e8a23690dbb, 10, Finished, Available, Finished, False)

In [10]:
display(
spark.sql("""
SELECT *
FROM bronze_healthcare_raw
LIMIT 10
""")
)

StatementMeta(, b4b70097-a892-4895-9852-0e8a23690dbb, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b277a4be-0e73-47f9-85ae-0212116553b1)

In [11]:
from pyspark.sql.functions import *

df_silver = spark.table("bronze_healthcare_raw")
print("Rows before:", df_silver.count())

df_silver = df_silver.dropDuplicates()
print("Rows after:", df_silver.count())


StatementMeta(, b4b70097-a892-4895-9852-0e8a23690dbb, 13, Finished, Available, Finished, False)

Rows before: 55500
Rows after: 54966


In [12]:
df_silver = df_silver.withColumn(
    "Name",
    initcap(col("Name"))
)

StatementMeta(, b4b70097-a892-4895-9852-0e8a23690dbb, 14, Finished, Available, Finished, False)

In [13]:
df_silver = df_silver.withColumn(
    "Length_of_Stay",
    datediff(col("Discharge_Date"), col("Date_of_Admission"))
)

StatementMeta(, b4b70097-a892-4895-9852-0e8a23690dbb, 15, Finished, Available, Finished, False)

In [14]:
from pyspark.sql.window import Window

df_silver = df_silver.withColumn(
    "Admission_ID",
    row_number().over(Window.orderBy(monotonically_increasing_id()))
)

StatementMeta(, b4b70097-a892-4895-9852-0e8a23690dbb, 16, Finished, Available, Finished, False)

In [15]:
cols = ["Admission_ID"] + [c for c in df_silver.columns if c != "Admission_ID"]

df_silver = df_silver.select(*cols)

StatementMeta(, b4b70097-a892-4895-9852-0e8a23690dbb, 17, Finished, Available, Finished, False)

In [16]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_healthcare_clean")

StatementMeta(, b4b70097-a892-4895-9852-0e8a23690dbb, 18, Finished, Available, Finished, False)

In [17]:
display(spark.table("silver_healthcare_clean"))

StatementMeta(, b4b70097-a892-4895-9852-0e8a23690dbb, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1c151a85-cd8d-45d7-81b0-18c1ed812283)

In [ ]:
df_gold = spark.table("silver_healthcare_clean")

StatementMeta(, b4b70097-a892-4895-9852-0e8a23690dbb, 20, Finished, Available, Finished, False)

In [ ]:
from pyspark.sql.functions import md5, concat_ws

dim_patient = df_gold.select(
    "Name",
    "Age",
    "Gender",
    "Blood_Type"
).dropDuplicates()

dim_patient = dim_patient.withColumn(
    "Patient_ID",
    md5(concat_ws("|", "Name", "Age", "Gender", "Blood_Type"))
)

display(dim_patient)

StatementMeta(, b4b70097-a892-4895-9852-0e8a23690dbb, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 73dc401e-8e5c-4ec1-9158-91e291612f66)

In [ ]:
dim_patient.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_patient")

StatementMeta(, b4b70097-a892-4895-9852-0e8a23690dbb, 22, Finished, Available, Finished, False)

In [ ]:
dim_doctor = df_gold.select(
    "Doctor"
).dropDuplicates()

dim_doctor = dim_doctor.withColumn(
    "Doctor_ID",
    md5(col("Doctor"))
)

display(dim_doctor)

StatementMeta(, b4b70097-a892-4895-9852-0e8a23690dbb, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c091be6a-d896-41b2-b3a8-696805db0300)

In [ ]:
dim_doctor.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_doctor")

StatementMeta(, b4b70097-a892-4895-9852-0e8a23690dbb, 24, Finished, Available, Finished, False)

In [ ]:
dim_hospital = df_gold.select(
    "Hospital"
).dropDuplicates()

dim_hospital = dim_hospital.withColumn(
    "Hospital_ID",
    md5(col("Hospital"))
)

StatementMeta(, b4b70097-a892-4895-9852-0e8a23690dbb, 25, Finished, Available, Finished, False)

In [33]:
dim_hospital.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_hospital")

StatementMeta(, b4b70097-a892-4895-9852-0e8a23690dbb, 35, Finished, Available, Finished, False)

In [ ]:
dim_insurance = df_gold.select(
    "Insurance_Provider"
).dropDuplicates()

dim_insurance = dim_insurance.withColumn(
    "Insurance_ID",
    md5(col("Insurance_Provider"))
)

StatementMeta(, b4b70097-a892-4895-9852-0e8a23690dbb, 26, Finished, Available, Finished, False)

In [ ]:
dim_insurance.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_insurance")

StatementMeta(, b4b70097-a892-4895-9852-0e8a23690dbb, 27, Finished, Available, Finished, False)

In [ ]:
fact_admissions = df_gold

StatementMeta(, b4b70097-a892-4895-9852-0e8a23690dbb, 28, Finished, Available, Finished, False)

In [ ]:
fact_admissions = fact_admissions.withColumn(
    "Patient_ID",
    md5(concat_ws("|", "Name", "Age", "Gender", "Blood_Type"))
)

StatementMeta(, b4b70097-a892-4895-9852-0e8a23690dbb, 29, Finished, Available, Finished, False)

In [ ]:
fact_admissions = fact_admissions.withColumn(
    "Doctor_ID",
    md5(col("Doctor"))
)

StatementMeta(, b4b70097-a892-4895-9852-0e8a23690dbb, 30, Finished, Available, Finished, False)

In [ ]:
fact_admissions = fact_admissions.withColumn(
    "Hospital_ID",
    md5(col("Hospital"))
)

StatementMeta(, b4b70097-a892-4895-9852-0e8a23690dbb, 31, Finished, Available, Finished, False)

In [ ]:
fact_admissions = fact_admissions.withColumn(
    "Insurance_ID",
    md5(col("Insurance_Provider"))
)

StatementMeta(, b4b70097-a892-4895-9852-0e8a23690dbb, 32, Finished, Available, Finished, False)

In [ ]:
fact_admissions = fact_admissions.select(
    "Admission_ID",
    "Patient_ID",
    "Doctor_ID",
    "Hospital_ID",
    "Insurance_ID",
    "Medical_Condition",
    "Medication",
    "Admission_Type",
    "Test_Results",
    "Billing_Amount",
    "Length_of_Stay",
    "Date_of_Admission",
    "Discharge_Date",
    "Room_Number"
)

StatementMeta(, b4b70097-a892-4895-9852-0e8a23690dbb, 33, Finished, Available, Finished, False)

In [ ]:
fact_admissions.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("fact_admissions")

StatementMeta(, b4b70097-a892-4895-9852-0e8a23690dbb, 34, Finished, Available, Finished, False)